# LLM-Based Text Watermark Detection using Llama 3.2 + NLP

**LLM:** Ollama `llama3.2`  
**NLP:** TF-IDF, n-grams, stylometric features  
**Classifier:** Logistic Regression  
**Metrics:** Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix  
**Interface:** Jupyter Notebook — no Streamlit, no Hugging Face.

> The supplied JSONL contains `id` and `text`, but no watermark labels. This notebook therefore creates an experimental synthetic detection dataset from Llama 3.2 outputs. The resulting scores are for this experimental setup.


## 1. Install dependencies

In [ ]:
!pip install -q ollama pandas numpy scipy scikit-learn nltk matplotlib seaborn tqdm joblib


## 2. Imports

In [ ]:
import json
import re
import math
import hashlib
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ollama

from tqdm.auto import tqdm
from scipy.sparse import csr_matrix, hstack

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)


## 3. Configure Ollama + Llama 3.2

Run `ollama pull llama3.2` in your terminal first.

In [ ]:
LLM_MODEL = "llama3.2"
print("Using LLM:", LLM_MODEL)


## 4. Load the JSONL dataset

In [ ]:
DATA_PATH = "data/train.jsonl"

records = []

with open(DATA_PATH, "r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


## 5. Text preprocessing

In [ ]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df = df.dropna(subset=["text"]).copy()
df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.len() > 0].reset_index(drop=True)

print("Usable records:", len(df))


## 6. Basic statistics

In [ ]:
df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))
df["char_count"] = df["clean_text"].apply(len)

display(df[["word_count", "char_count"]].describe())


## 7. Llama 3.2 generation function

In [ ]:
def generate_llama_text(prompt, max_tokens=120):
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0.7,
            "top_p": 0.9,
            "num_predict": max_tokens
        }
    )
    return response["message"]["content"]


## 8. Test Llama 3.2

In [ ]:
prompt = df.iloc[0]["clean_text"][:500]

generated_text = generate_llama_text(
    prompt,
    max_tokens=120
)

print(generated_text)


## 9. Deterministic lexical watermark signal

Ollama's standard chat interface does not expose Llama token logits. This notebook therefore uses a deterministic lexical watermark signal. A secret key assigns words to a green list, and the detector measures the resulting statistical signal.


In [ ]:
WATERMARK_KEY = "LLAMA32_NLP_WATERMARK_2026"
GREEN_FRACTION = 0.5

def is_green_word(word, key=WATERMARK_KEY):
    word = word.lower().strip()

    if not word.isalpha():
        return False

    digest = hashlib.sha256(
        f"{key}:{word}".encode("utf-8")
    ).hexdigest()

    number = int(digest, 16)

    return (number % 1_000_000) / 1_000_000 < GREEN_FRACTION


## 10. Watermark z-score detector

In [ ]:
def calculate_watermark_score(text, key=WATERMARK_KEY):
    words = re.findall(r"[A-Za-z]+", text.lower())

    if len(words) < 5:
        return {
            "z_score": 0.0,
            "green_count": 0,
            "total_words": len(words),
            "green_ratio": 0.0
        }

    green_count = sum(
        is_green_word(word, key)
        for word in words
    )

    n = len(words)
    ratio = green_count / n
    expected = n * GREEN_FRACTION
    variance = n * GREEN_FRACTION * (1 - GREEN_FRACTION)

    z_score = (green_count - expected) / math.sqrt(variance)

    return {
        "z_score": z_score,
        "green_count": green_count,
        "total_words": n,
        "green_ratio": ratio
    }


## 11. Test watermark score

In [ ]:
calculate_watermark_score(generated_text)


## 12. Create experimental labeled data

`0` = normal Llama 3.2 generation  
`1` = Llama 3.2 generation prompted to favor the secret watermark vocabulary signal.

These are synthetic experimental labels because the source JSONL does not contain watermark labels.


In [ ]:
SAMPLE_SIZE = min(50, len(df))

generated_records = []

for i in tqdm(range(SAMPLE_SIZE)):
    source = df.iloc[i]["clean_text"][:500]

    normal_text = generate_llama_text(
        f"Write a natural response based on this text:\n{source}",
        max_tokens=120
    )

    generated_records.append({
        "source_id": df.iloc[i]["id"],
        "text": normal_text,
        "label": 0,
        "generation_type": "normal"
    })

    watermark_prompt = f"""
Write a natural response based on the text below.

Use vocabulary choices that favor words associated with
this secret watermark key: {WATERMARK_KEY}

Do not mention the key or watermark.
Keep the response coherent and natural.

Source:
{source}
"""

    watermarked_text = generate_llama_text(
        watermark_prompt,
        max_tokens=120
    )

    generated_records.append({
        "source_id": df.iloc[i]["id"],
        "text": watermarked_text,
        "label": 1,
        "generation_type": "watermarked"
    })

detection_df = pd.DataFrame(generated_records)

print("Detection dataset:", detection_df.shape)
print(detection_df["label"].value_counts())


## 13. Extract watermark features

In [ ]:
def extract_watermark_features(text):
    result = calculate_watermark_score(text)

    return [
        result["z_score"],
        result["green_ratio"],
        result["total_words"]
    ]

watermark_features = np.array([
    extract_watermark_features(text)
    for text in detection_df["text"]
])

print("Shape:", watermark_features.shape)


## 14. Extract stylometric NLP features

In [ ]:
def stylometric_features(text):
    words = re.findall(r"[A-Za-z]+", text)

    if not words:
        return [0, 0, 0, 0, 0, 0]

    sentences = [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

    word_lengths = [len(w) for w in words]

    unique_ratio = (
        len(set(w.lower() for w in words)) / len(words)
    )

    avg_word_length = np.mean(word_lengths)

    avg_sentence_length = (
        len(words) / max(len(sentences), 1)
    )

    punctuation_count = len(
        re.findall(r"[,;:!?]", text)
    )

    uppercase_count = len(
        re.findall(r"\b[A-Z][a-z]+\b", text)
    )

    return [
        len(words),
        avg_word_length,
        unique_ratio,
        avg_sentence_length,
        punctuation_count,
        uppercase_count
    ]

style_features = np.array([
    stylometric_features(text)
    for text in detection_df["text"]
])

print("Shape:", style_features.shape)


## 15. TF-IDF features

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)

X_tfidf = tfidf_vectorizer.fit_transform(
    detection_df["text"]
)

print("TF-IDF shape:", X_tfidf.shape)


## 16. Feature fusion

In [ ]:
X_extra = np.hstack([
    watermark_features,
    style_features
])

X = hstack([
    X_tfidf,
    csr_matrix(X_extra)
])

y = detection_df["label"].values

print("Final feature matrix:", X.shape)


## 17. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape[0])
print("Testing :", X_test.shape[0])


## 18. Train Logistic Regression

In [ ]:
classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

classifier.fit(X_train, y_train)

print("Training completed.")


## 19. Predictions

In [ ]:
y_pred = classifier.predict(X_test)
y_probability = classifier.predict_proba(X_test)[:, 1]


## 20. Evaluation metrics

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_probability)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics)


## 21. Classification report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Not Watermarked", "Watermarked"],
        zero_division=0
    )
)


## 22. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["Not Watermarked", "Watermarked"],
    yticklabels=["Not Watermarked", "Watermarked"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Llama 3.2 Watermark Detection")
plt.show()


## 23. Final inference function

In [ ]:
def final_detector(text):
    tfidf_features = tfidf_vectorizer.transform([text])

    wm_features = np.array([
        extract_watermark_features(text)
    ])

    style = np.array([
        stylometric_features(text)
    ])

    extra = csr_matrix(
        np.hstack([wm_features, style])
    )

    final_features = hstack([
        tfidf_features,
        extra
    ])

    probability = classifier.predict_proba(
        final_features
    )[0, 1]

    wm_score = calculate_watermark_score(text)

    prediction = (
        "Watermarked"
        if probability >= 0.5
        else "Not Watermarked"
    )

    return {
        "prediction": prediction,
        "probability": float(probability),
        "z_score": float(wm_score["z_score"]),
        "green_ratio": float(wm_score["green_ratio"]),
        "total_words": int(wm_score["total_words"])
    }


## 24. Test detector

In [ ]:
result = final_detector(generated_text)

for key, value in result.items():
    print(f"{key}: {value}")


## 25. Generate new Llama 3.2 text and detect it

In [ ]:
new_prompt = """
Explain how artificial intelligence is changing education.
"""

new_text = generate_llama_text(
    new_prompt,
    max_tokens=150
)

print("Llama 3.2 output:")
print("-" * 70)
print(new_text)

print("\nDetection:")
print("-" * 70)

result = final_detector(new_text)

for key, value in result.items():
    print(f"{key}: {value}")


## 26. Save the trained NLP model

In [ ]:
joblib.dump(
    classifier,
    "llama32_nlp_watermark_classifier.pkl"
)

joblib.dump(
    tfidf_vectorizer,
    "llama32_tfidf_vectorizer.pkl"
)

print("Models saved successfully.")


# Final Project Architecture

JSONL → preprocessing → **Ollama Llama 3.2** → normal/watermark-oriented generation → **NLP features (TF-IDF + n-grams + stylometry + watermark statistics)** → Logistic Regression → Watermarked / Not Watermarked → Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix.

**No Hugging Face. No Streamlit.**

For a research-grade version, use ground-truth watermark labels and evaluate on human text, unwatermarked LLM text, genuinely watermarked LLM text, and paraphrased/adversarial text.
